In [1]:
import pandas as pd
import numpy as np

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error


# ─── Hamaara same Zomato dataset ───
data = {
    'Avg_Bill':       [350,   np.nan, 180,   np.nan, 120,   480  ],
    'Rating':         [4.2,   4.5,    np.nan, 3.8,   4.0,   np.nan],
    'Delivery_Time':  [30,    25,     40,     np.nan, 20,   35   ],
}
df = pd.DataFrame(data)

print("Original Data (NaN = missing):")
print(df)
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# ─── IterativeImputer — MICE ───
# Parameters:
#   max_iter  = kitne rounds chalega (10 enough hai)
#   random_state = reproducibility ke liye
imputer = IterativeImputer(max_iter=10, random_state=42)

# fit_transform = dataset pe MICE apply karo
# Ye ek numpy array return karta hai → DataFrame mein convert karo
df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

print("\nAfter MICE Imputation (sabhi values fill ho gayi!):")
display(df_imputed.round(2))

print(f"\nTotal missing values after: {df_imputed.isnull().sum().sum()}")
# → 0! Koi NaN nahi baka!

# ─── Compare karo: Mean vs MICE ───
print("\n--- Comparison: Mean vs MICE ---")
mean_fill = df.fillna(df.mean())

print("\nPizzaPalace Avg_Bill:")
print(f"  Mean fill  → ₹{mean_fill.loc[1, 'Avg_Bill']:.0f}  (overall average)")
print(f"  MICE fill  → ₹{df_imputed.loc[1, 'Avg_Bill']:.0f}  (context-aware!)")

print("\nBurgerZone Avg_Bill:")
print(f"  Mean fill  → ₹{mean_fill.loc[3, 'Avg_Bill']:.0f}  (same as PizzaPalace — wrong!)")
print(f"  MICE fill  → ₹{df_imputed.loc[3, 'Avg_Bill']:.0f}  (lower — BurgerZone ka rating kam hai)")

Original Data (NaN = missing):
   Avg_Bill  Rating  Delivery_Time
0     350.0     4.2           30.0
1       NaN     4.5           25.0
2     180.0     NaN           40.0
3       NaN     3.8            NaN
4     120.0     4.0           20.0
5     480.0     NaN           35.0

Total missing values: 5

After MICE Imputation (sabhi values fill ho gayi!):


,Avg_Bill,Rating,Delivery_Time
0,350.00,4.20,30.0
1,282.49,4.50,25.0
2,180.00,1.73,40.0
3,282.50,3.80,30.0
4,120.00,4.00,20.0
5,480.00,4.46,35.0



Total missing values after: 0

--- Comparison: Mean vs MICE ---

PizzaPalace Avg_Bill:
  Mean fill  → ₹282  (overall average)
  MICE fill  → ₹282  (context-aware!)

BurgerZone Avg_Bill:
  Mean fill  → ₹282  (same as PizzaPalace — wrong!)
  MICE fill  → ₹282  (lower — BurgerZone ka rating kam hai)


In [ ]:
np.random.seed(42)
n = 500

X1 = np.random.normal(50, 10, n)
X2 = X1 * 0.7 + np.random.normal(0, 5, n)
X3 = np.random.normal(30, 8, n)
X4 = 0.6 * X1 + 0.3 * X2 + np.random.normal(0, 4, n)

df = pd.DataFrame({
    "X1": X1,
    "X2": X2,
    "X3": X3,
    "X4": X4
})

median = df["X1"].median()
mask = df["X1"] < median
mar_index = df[mask].sample(frac=0.30, random_state=42).index
true_values = df.loc[mar_index, "X4"].copy()

df.loc[mar_index, "X4"] = np.nan

print("Missing Values:")
print(df.isnull().sum())

imputer = IterativeImputer(random_state=42)
imputed = imputer.fit_transform(df)

df_imputed = pd.DataFrame(imputed, columns=df.columns)

predicted = df_imputed.loc[mar_index, "X4"]

rmse = np.sqrt(mean_squared_error(true_values, predicted))

print("\nRMSE:", rmse)

Missing Values:
X1     0
X2     0
X3     0
X4    75
dtype: int64

RMSE: 4.045106661686583


In [3]:
from sklearn.preprocessing import OrdinalEncoder


np.random.seed(42)
n = 500

df = pd.DataFrame({
    "Age": np.random.randint(18, 60, n),
    "Salary": np.random.randint(20000, 100000, n),
    "Experience": np.random.randint(1, 20, n),
    "Gender": np.random.choice(["Male", "Female"], n),
    "Department": np.random.choice(["HR", "IT", "Sales"], n)
})

num_cols = ["Age", "Salary", "Experience"]
cat_cols = ["Gender", "Department"]
    
encoder = OrdinalEncoder()
df[cat_cols] = encoder.fit_transform(df[cat_cols])


for col in df.columns:
    df.loc[df.sample(frac=0.15, random_state=42).index, col] = np.nan

print("Missing Values:")
print(df.isnull().sum())


imputer = IterativeImputer(random_state=42)
imputed = imputer.fit_transform(df)


df_imputed = pd.DataFrame(imputed, columns=df.columns)


for col in cat_cols:
    df_imputed[col] = np.round(df_imputed[col])


df_imputed[cat_cols] = encoder.inverse_transform(df_imputed[cat_cols])

print("\nGender Distribution:")
print(df_imputed["Gender"].value_counts())

print("\nDepartment Distribution:")

print(df_imputed["Department"].value_counts())

print("\nFinal Dataset:")
print(df_imputed.head())

Missing Values:
Age           75
Salary        75
Experience    75
Gender        75
Department    75
dtype: int64

Gender Distribution:
Gender
Female    303
Male      197
Name: count, dtype: int64

Department Distribution:
Department
IT       223
HR       151
Sales    126
Name: count, dtype: int64

Final Dataset:
         Age        Salary  Experience  Gender Department
0  39.527059  58210.115294   10.171765  Female         IT
1  46.000000  64811.000000    4.000000  Female      Sales
2  39.527059  58210.115294   10.171765  Female         IT
3  25.000000  40150.000000    5.000000    Male         IT
4  38.000000  91180.000000    8.000000    Male         IT
